# Chapter 7 — Worked Example: Recharge Estimation, Oued Sikkak Watershed

**AI for Hydrogeologists** — companion notebook

The Oued Sikkak watershed (218 km2, NW Algeria) contains the Hennaya plain
studied in earlier chapters. Basin facts (Bouanani et al., 2013, *Journal of
Water Science*): area 218 km2, mean annual precipitation 480.5 mm
(206-800 mm range), mean annual runoff ~104 mm.

**Scope note on GRACE-FO.** The book outline mentions GRACE-FO storage
anomalies for recharge estimation. GRACE's native footprint spans several
hundred km — far coarser than this 218 km2 basin (the same resolution
mismatch identified for GLiM in Chapter 5). We instead use the actual
Zenata climate station record (used in the published literature for this
exact basin) to compute a monthly climatic water balance as a recharge
**proxy** — a first-order, uncalibrated estimate, not a validated recharge
model.

In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

BASE = "https://raw.githubusercontent.com/Dr-LAOUFIAbdessalam/ai-hydrogeologists/main/"
precip = pd.read_excel(BASE + "ch02_data_preparation/data/raw/Precipitations_mensuelles.xlsx")
temp = pd.read_excel(BASE + "ch07_recharge/data/raw/Temperatures_mensuelles.xlsx")
print(precip.shape, temp.shape)


(45, 13) (45, 13)


## Reshape to a chronological monthly series

In [23]:
months_fr = ["Sep","Oct","Nov","Dec","Jan","Fev","Mars","Avr","Mai","Juin","Juill","Aout"]
months_en = ["Sep","Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug"]
month_num = {"Sep":9,"Oct":10,"Nov":11,"Dec":12,"Jan":1,"Fev":2,"Feb":2,"Mars":3,"Mar":3,
             "Avr":4,"Apr":4,"Mai":5,"May":5,"Juin":6,"Jun":6,"Juill":7,"Jul":7,"Aout":8,"Aug":8}

p_long = precip.melt(id_vars="Année_hydro", value_vars=months_fr, var_name="month_fr", value_name="P_mm")
t_long = temp.melt(id_vars="Année_hydro", value_vars=months_en, var_name="month_en", value_name="T_C")
p_long["month_num"] = p_long["month_fr"].map(month_num)
t_long["month_num"] = t_long["month_en"].map(month_num)

df = p_long.merge(t_long, on=["Année_hydro", "month_num"])
df["hydro_year_start"] = df["Année_hydro"].str.split("/").str[0].astype(int)

def to_date(row):
    year = row["hydro_year_start"] if row["month_num"] >= 9 else row["hydro_year_start"] + 1
    return pd.Timestamp(year=year, month=row["month_num"], day=1)
df["date"] = df.apply(to_date, axis=1)
df = df.sort_values("date").reset_index(drop=True)

# Drop the incomplete first hydrological year (established in Chapter 2)
df = df[df["hydro_year_start"] >= 1981].reset_index(drop=True)
date_min = df['date'].min().date()
date_max = df['date'].max().date()
print(f"n = {len(df)} months, {date_min} to {date_max}")


n = 528 months, 1981-09-01 to 2025-08-01


## Thornthwaite potential evapotranspiration (Zenata, lat ~34.98 N)

In [24]:
LAT = 34.98

def day_length_factor(month, lat=LAT):
    lat_rad = np.radians(lat)
    day_of_year = {1:15,2:46,3:74,4:105,5:135,6:162,7:198,8:228,9:259,10:289,11:320,12:350}[month]
    decl = 0.4093 * np.sin(2*np.pi*day_of_year/365 - 1.405)
    ws = np.arccos(-np.tan(lat_rad) * np.tan(decl))
    N = 24/np.pi * ws
    days_in_month = {1:31,2:28,3:31,4:30,5:31,6:30,7:31,8:31,9:30,10:31,11:30,12:31}[month]
    return (N/12) * (days_in_month/30)

annual = df.groupby("hydro_year_start")["T_C"].apply(lambda t: np.sum((np.maximum(t, 0)/5)**1.514))
df["I_annual"] = df["hydro_year_start"].map(annual)
a = (6.75e-7*df["I_annual"]**3) - (7.71e-5*df["I_annual"]**2) + (1.792e-2*df["I_annual"]) + 0.49239

T_pos = np.maximum(df["T_C"], 0)
pet_unadjusted = 16 * (10*T_pos/df["I_annual"])**a
df["PET_mm"] = pet_unadjusted * df["month_num"].apply(day_length_factor)
df.loc[df["T_C"] <= 0, "PET_mm"] = 0


## Monthly water balance and comparison with published runoff

In [25]:
df["WB_surplus_mm"] = np.maximum(df["P_mm"] - df["PET_mm"], 0)

annual_summary = df.groupby("hydro_year_start").agg(
    P_annual=("P_mm","sum"), PET_annual=("PET_mm","sum"), surplus_annual=("WB_surplus_mm","sum"))
print("Annual water balance (mean over record):")
print(annual_summary.mean().round(1))
print("\nPublished mean annual runoff, Sikkak basin (Bouanani et al. 2013): 104 mm")


Annual water balance (mean over record):
P_annual          311.1
PET_annual        895.3
surplus_annual    102.8
dtype: float64

Published mean annual runoff, Sikkak basin (Bouanani et al. 2013): 104 mm


**Result:** the water balance surplus (~103 mm/yr) closely matches the
independently published mean annual runoff (104 mm/yr) — a striking
plausibility check for a completely uncalibrated model, though this
surplus represents runoff **and** recharge combined, not recharge alone,
and the close numerical match should not be over-interpreted as proof of
model accuracy given the simplicity of the bucket approach used.

## Feature engineering: lag structure

In [26]:
for lag in [1, 2, 3, 6, 12]:
    df[f"P_lag{lag}"] = df["P_mm"].shift(lag)
    df[f"T_lag{lag}"] = df["T_C"].shift(lag)
df["P_cumul_3mo"] = df["P_mm"].rolling(3).sum()
df["P_cumul_6mo"] = df["P_mm"].rolling(6).sum()
df["month_sin"] = np.sin(2*np.pi*df["month_num"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month_num"]/12)

model_df = df.dropna().reset_index(drop=True)
feature_cols = [c for c in model_df.columns if c.startswith(("P_lag","T_lag","P_cumul","month_sin","month_cos"))]
X = model_df[feature_cols].values
y = model_df["WB_surplus_mm"].values
print(f"n={len(model_df)} months, {len(feature_cols)} features")


n=516 months, 14 features


## Random split vs year-blocked temporal split (Section 3.5.1/3.5.3)

In [27]:
years = model_df["hydro_year_start"].values
unique_years = np.sort(np.unique(years))
year_folds = np.array_split(unique_years, 6)

def temporal_cv(model, X, y, years, year_folds):
    rmses, r2s = [], []
    for fold_years in year_folds:
        test_idx = np.isin(years, fold_years)
        train_idx = ~test_idx
        if test_idx.sum() < 5:
            continue
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])
        rmses.append(np.sqrt(mean_squared_error(y[test_idx], pred)))
        r2s.append(r2_score(y[test_idx], pred))
    return np.array(rmses), np.array(r2s)

def random_cv(model, X, y, n_splits=6, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    folds = np.array_split(idx, n_splits)
    rmses, r2s = [], []
    for i in range(n_splits):
        test_idx = folds[i]
        train_idx = np.hstack([folds[j] for j in range(n_splits) if j != i])
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])
        rmses.append(np.sqrt(mean_squared_error(y[test_idx], pred)))
        r2s.append(r2_score(y[test_idx], pred))
    return np.array(rmses), np.array(r2s)

models = {"Linear Regression": LinearRegression(),
          "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)}

for name, model in models.items():
    rmse_r, r2_r = random_cv(model, X, y)
    rmse_t, r2_t = temporal_cv(model, X, y, years, year_folds)
    print(f"\n{name}:")
    print(f"  Random CV   : RMSE={rmse_r.mean():.1f}+/-{rmse_r.std():.1f} mm | R2={r2_r.mean():.2f}")
    print(f"  Temporal CV : RMSE={rmse_t.mean():.1f}+/-{rmse_t.std():.1f} mm | R2={r2_t.mean():.2f}")



Linear Regression:
  Random CV   : RMSE=9.9+/-0.8 mm | R2=0.74
  Temporal CV : RMSE=10.0+/-1.4 mm | R2=0.70

Random Forest:
  Random CV   : RMSE=11.7+/-1.7 mm | R2=0.65
  Temporal CV : RMSE=12.1+/-3.0 mm | R2=0.59


**Interpretation:** the random-vs-temporal CV gap here (R2 ~0.70->0.65 for
linear regression) is much smaller than the catastrophic collapse seen for
the spatial case in Chapter 3 (R2 ~0.70 -> -0.40). Not every naive-split
problem fails equally badly — the degree of optimistic bias depends on the
correlation structure of the specific problem; strong seasonality here
lets both random and temporal splits capture much of the signal. Note also
that the simple linear model slightly outperforms Random Forest, a useful
reminder that model complexity is not automatically an advantage.

## Feature importance

In [28]:
rf_full = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42).fit(X, y)
importances = pd.Series(rf_full.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(8).round(3))


P_cumul_3mo    0.463
P_lag2         0.131
P_lag1         0.130
T_lag6         0.054
T_lag12        0.041
T_lag3         0.031
month_cos      0.024
P_lag12        0.023
dtype: float64


The 3-month cumulative precipitation dominates (importance ~0.46),
consistent with a delayed recharge response to antecedent rainfall rather
than the current month alone — matching the physical expectation
established in Chapter 2 (Section 2.4.1, lag variables).

## Partial validation: field water-point inventory (22 locations)

A field inventory of 22 water points across the Sikkak watershed (8
natural springs, 14 wells) offers a partial, real-world check — but
springs and wells play fundamentally different roles and must not be
conflated.

In [29]:
BASE2 = "https://raw.githubusercontent.com/Dr-LAOUFIAbdessalam/ai-hydrogeologists/main/"
wp = pd.read_csv(BASE2 + "ch07_recharge/data/raw/points_d_eau_sikkak.csv")
wp["type"] = wp["Source"].apply(lambda s: "spring" if s.startswith("Ain") else "well")

BASIN_AREA_KM2 = 218
SECONDS_PER_YEAR = 86400 * 365.25

def flow_to_mm_per_year(Q_Ls_sum, area_km2=BASIN_AREA_KM2):
    Q_m3s = Q_Ls_sum / 1000
    annual_m3 = Q_m3s * SECONDS_PER_YEAR
    return annual_m3 / (area_km2 * 1e6) * 1000

spring_sum = wp.loc[wp.type == "spring", "Q (l/s)"].sum()
well_sum = wp.loc[wp.type == "well", "Q (l/s)"].sum()
spring_mm = flow_to_mm_per_year(spring_sum)
well_mm = flow_to_mm_per_year(well_sum)
surplus_mean = annual_summary["surplus_annual"].mean()

n_springs = sum(wp.type == "spring")
n_wells = sum(wp.type == "well")
print(f"Springs (n={n_springs}): {spring_sum:.1f} L/s -> {spring_mm:.1f} mm/yr (natural baseflow)")
print(f"Wells (n={n_wells}): {well_sum:.1f} L/s -> {well_mm:.1f} mm/yr (measured extraction)")
print(f"Water balance surplus (this notebook): {surplus_mean:.1f} mm/yr")
print(f"\nExploitation ratio (partial wells / surplus): {100*well_mm/surplus_mean:.0f}% (lower bound, incomplete well inventory)")


Springs (n=8): 89.5 L/s -> 13.0 mm/yr (natural baseflow)
Wells (n=14): 245.0 L/s -> 35.5 mm/yr (measured extraction)
Water balance surplus (this notebook): 102.8 mm/yr

Exploitation ratio (partial wells / surplus): 34% (lower bound, incomplete well inventory)


**Interpretation:**
- **Springs (13.0 mm/yr) < water balance surplus (102.8 mm/yr)** — this is
  the expected direction and therefore a genuine (if partial) consistency
  check: natural spring baseflow is only one discharge component of the
  total surplus, which also includes direct runoff and water intercepted
  by pumping before ever reaching a spring.
- **Wells measure human extraction, not natural recharge.** Using them to
  "validate" the recharge estimate would be a category error. They are,
  however, informative for a separate and practically important question:
  what fraction of estimated recharge is currently being extracted? The
  naive ratio here (~34%) is necessarily a **lower bound**, since these
  14 wells are a partial inventory, not a census of all abstraction in
  the 218 km2 basin.